In [1]:
import os
from pathlib import Path

os.chdir(Path(os.getcwd()).parent.parent)
print(os.getcwd())

/home/prateek/Documents/projects/headpose-estimation


In [2]:
from headpose_estimation.utils.tensorflow_model_handler import TensorflowV1ModelHandler

tf_model_handler = TensorflowV1ModelHandler(
  tf_models_config_path="configs/tf_models/tf_models.yaml", model_storage_path="models"
)

tf_model_handler.get_model(architecture="inception_v3")
tf_model_config = tf_model_handler.get_model_info(architecture="inception_v3")

In [3]:
import tensorflow as tf

from headpose_estimation.models import (
  EulerAnglesPredictionHead,
  EulerAnglesPredictionModel,
  PretrainedBackBoneImageModel,
)
from headpose_estimation.utils.tensorflow_model_handler import TensorflowV1ModelHandler

sess = tf.Session()
backbone_model = PretrainedBackBoneImageModel(tf_model_config=tf_model_config, session=sess)


2026-05-17 12:39:06.937279: W tensorflow/stream_executor/platform/default/dso_loader.cc:55] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-05-17 12:39:06.937308: E tensorflow/stream_executor/cuda/cuda_driver.cc:318] failed call to cuInit: UNKNOWN ERROR (303)
2026-05-17 12:39:06.937327: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (bleu): /proc/driver/nvidia/version does not exist
2026-05-17 12:39:06.937887: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2026-05-17 12:39:06.944878: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 1796820000 Hz
2026-05-17 12:39:06.945877: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x6431e64f5c40 initialized for platform Host (this does not guarantee that XLA w

In [ ]:
writer = tf.summary.FileWriter("models/inception_v3/logs/", sess.graph)
writer.close()

In [4]:
with tf.gfile.FastGFile("data/raw/300W-LP/300W_LP/AFW_Flip/AFW_134212_1_0.jpg", "rb") as f:
  image_data = f.read()

output = backbone_model.forward(image_data=[image_data])

print(type(output))
print(len(output))
print(output[0].shape)

<class 'list'>
1
(1, 2048)


In [5]:
angle_prediction_head = EulerAnglesPredictionHead(
  input_image_representation_tensor=tf.placeholder(
    tf.float32, shape=[None, backbone_model.model_config.output_tensor_size], name="angle_prediction_input"
  ),
  layer_sizes=[1024, 512, 1],
  session=sess,
)

sess.run(tf.global_variables_initializer())

In [8]:
print(output[0].shape)
print(np.vstack(output).shape)

(1, 2048)
(1, 2048)


In [6]:
import numpy as np

batched_input = np.vstack((output[0], output[0]))
print(batched_input.shape)

(2, 2048)


In [9]:
angle_prediction_output = angle_prediction_head.forward(image_input_representation=np.vstack(output))
print(angle_prediction_output.shape)

(1, 1)
(1, 1)
(1, 1)
(1, 3)
